# 第 7 章：部署、运维与综合实践

检索增强生成（RAG, Retrieval-Augmented Generation）系统上线后，问题不只在模型：知识索引何时更新、数据库是否可读、访问边界是否正确、故障时如何定位，都决定答案能否被信任。本章继续使用项目的自然语言转结构化查询语言（NL2SQL, Natural Language to SQL）助手，讨论如何让结构化查询语言（SQL, Structured Query Language）的生成与执行从本地样例走向可维护服务。智能体开发工具包（ADK, Agent Development Kit）负责管理工具调用，大语言模型（LLM, Large Language Model）提出查询，数据库提供实际数值。

**学习目标**：读懂现有配置和启动顺序；比较本地、单实例与托管部署；解释增量索引与模型迁移；做一份可复现的排障运行手册。默认单元仅检查仓库文件和执行离线演算，不读取 <code>.env</code>、不打印密钥、不连接数据库、不调用云服务。首次使用请在项目根目录运行 uv sync --extra dev，并用项目虚拟环境启动 Jupyter。数据库与在线实验均有显式开关。


## 7.1 场景：昨天还能回答，今天却说“知识索引为空”

Qwen 的向量嵌入（Embedding）把文本映射为可比较的向量。一条自然语言分析请求至少依赖三条链：业务库 <code>analytics</code> 有事实数据；知识库 <code>rag.documents</code> 有与当前模型匹配的向量；智能体（Agent）能调用检索与只读查询工具。即使数据库容器显示健康，也不能证明迁移、种子、索引和模型访问都正常。

~~~mermaid
flowchart LR
  K[db/knowledge.json + 实时字段] --> L[load_knowledge]
  L --> H[内容哈希与模型名比较]
  H --> E[变化文档调用 Qwen Embedding]
  E --> U[(rag.documents 增量写入)]
  Q[用户问题] --> A[ADK / DeepSeek]
  A --> R[Qwen 查询向量 + 检索]
  R --> U
  A --> S[只读 SQL]
  S --> B[(analytics 业务表)]
  U --> A
  B --> A
~~~

ADK 在本项目中暴露 <code>root_agent</code>。<code>adk web .</code> 是开发调试入口；<code>adk api_server .</code> 可作为接口服务入口。仓库的 <code>compose.yaml</code> **只启动 PostgreSQL/pgvector**，没有把 Agent 服务容器化，也没有现成的认证、租户隔离或调度任务；生产部署必须另行设计这些层。


In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "tutorial":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "请从项目根目录或 tutorial 目录启动 Notebook"

RUN_DB_READ = False
RUN_ONLINE_RETRIEVAL = False
print("项目根目录:", ROOT)
print("当前默认：不连接数据库，不请求云模型，不写入数据")


项目根目录: D:\Projects\PythonProject\nl2sql-rag
当前默认：不连接数据库，不请求云模型，不写入数据


## 7.2 三种部署落点如何选？

| 方案 | 适合阶段 | 优点 | 主要代价与工作 |
|---|---|---|---|
| 开发机：Docker Compose 数据库 + 本地 ADK | 课程、功能验证 | 环境简单、链路可见、方便逐层调试 | 单机状态不可代表生产可用性；样例密码仅用于本地 |
| 单实例服务：独立 Agent 进程 + PostgreSQL/pgvector | 小团队内部试用 | 可单独重启服务与数据库，易接入日志和备份 | 需要反向代理、认证、密钥管理、健康检查、升级回滚 |
| 托管应用 + 托管 PostgreSQL/搜索服务 | 流量、隔离和可用性要求较高 | 备份、弹性和监控可由平台承担一部分 | 成本、网络与数据驻留、服务配额、模型依赖都需评估 |

本项目把业务事实和知识向量放在同一个 PostgreSQL 实例，减少本地组件数量。规模上升后可把应用服务与数据库分离；是否要再拆独立向量库，应由数据量、过滤需求、延迟和运维能力决定。若知识只有 13 张卡，额外部署一套搜索集群通常得不到足以抵消复杂度的收益。

每问的大致调用成本可拆成：

$$C_{\mathrm{request}} = C_{\mathrm{query\ embedding}} + C_{\mathrm{retrieval}} + C_{\mathrm{LLM}} + C_{\mathrm{database}} + C_{\mathrm{optional\ rerank}}.$$

精排（Rerank）只在第 3 章的可选实验中出现，**未接入当前 Agent 的默认路径**。部署预算应以真实工具轨迹测量，不要把可选组件计成固定成本。


## 7.3 配置是可检查的契约

<code>pyproject.toml</code> 要求 Python 3.12 至 3.13，主要依赖 Google ADK、LiteLLM、DashScope、psycopg、pgvector、SQLGlot 和 pydantic-settings。<code>compose.yaml</code> 使用 PostgreSQL 17 的 pgvector 镜像。<code>src/nl2sql/config.py</code> 的配置来自进程环境和项目根目录的 <code>.env</code>；从其他工作目录启动时，需确认相对路径仍指向正确的配置文件。

管理连接 <code>DB_ADMIN_URL</code> 只供迁移、种子和索引写入使用；运行期检索与 SQL 执行使用只读 <code>DB_READER_URL</code>。Qwen Embedding 使用 DashScope 原生应用程序编程接口（API, Application Programming Interface），<code>DASHSCOPE_BASE_URL</code> 应指向原生 <code>/api/v1</code> 路径。仓库的 <code>.env.example</code> 是字段样板；真实凭据应由环境变量或受控密钥存储提供，不能写进 Notebook 输出或版本库。

下一格**只读取样板中的字段名**，不读取 <code>.env</code> 或任何字段值。


In [2]:
import tomllib

project = tomllib.loads((ROOT / "pyproject.toml").read_text(encoding="utf-8"))["project"]
sample_keys = [
    line.split("=", 1)[0].strip()
    for line in (ROOT / ".env.example").read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#") and "=" in line
]
print("Python 要求:", project["requires-python"])
print("配置字段名:", sample_keys)
assert {"DB_ADMIN_URL", "DB_READER_URL", "QWEN_EMBED_DIM", "DASHSCOPE_BASE_URL"} <= set(sample_keys)


Python 要求: >=3.12,<3.14
配置字段名: ['POSTGRES_USER', 'POSTGRES_PASSWORD', 'POSTGRES_DB', 'POSTGRES_PORT', 'DB_ADMIN_URL', 'DB_READER_URL', 'DB_READER_PASSWORD', 'DEEPSEEK_API_KEY', 'DEEPSEEK_MODEL', 'QWEN_API_KEY', 'QWEN_EMBED_MODEL', 'QWEN_EMBED_DIM', 'DASHSCOPE_BASE_URL']


### 维度、模型和数据库列必须一致

<code>db/schema.sql</code> 将向量列定义为 <code>vector(1024)</code>，并创建用于余弦搜索的分层可导航小世界索引（HNSW, Hierarchical Navigable Small World）。<code>nl2sql.migrate.migrate()</code> 也显式检查配置维度为 1024。索引器记录 <code>embed_model</code>，检索只查同一模型的文档；仅改模型名而不重建向量会导致索引不可用。换到其他维度需要先设计数据库迁移和新索引，再在代表性问题集上比较召回质量。13 张卡的实验不能推断大语料的索引性能。


In [3]:
import re

schema_sql = (ROOT / "db/schema.sql").read_text(encoding="utf-8")
dimension = int(re.search(r"embedding\s+vector\((\d+)\)", schema_sql).group(1))
index_present = "USING hnsw" in schema_sql and "vector_cosine_ops" in schema_sql
print({"向量列维度": dimension, "HNSW 余弦索引已声明": index_present})
assert dimension == 1024 and index_present


{'向量列维度': 1024, 'HNSW 余弦索引已声明': True}


## 7.4 增量索引：变更的最小单位是知识卡

每张知识卡拥有稳定的文档标识符（ID, Identifier），用于更新、删除和引用。

<code>nl2sql.knowledge.load_knowledge()</code> 读取审核过的 <code>db/knowledge.json</code>；对表卡还会拼入 <code>information_schema</code> 的实时字段。<code>KnowledgeDoc.embedding_text</code> 汇总类型、对象、别名和正文，<code>content_hash</code> 对这段实际嵌入文本做哈希。<code>index_knowledge()</code> 用 <code>(content_hash, embed_model)</code> 判断是否需要重新向量化，再在数据库事务中更新变化文档、删除源中已不存在的 ID。

这个办法适合单一、很小且可信的知识源。它每次仍要读取全部源和数据库字段、对比全部 ID；大量文档时可以采用变更事件、队列和批次发布。生产版还应有索引版本、构建完成校验、回滚指针和来源删除审计。当前代码中的 <code>source_version</code> 固定为 <code>catalog-v1</code>，尚未实现双版本切换。

下面只构造内存中的指标卡，验证改别名会改变哈希而稳定 ID 不变；不会调用 <code>load_knowledge()</code>，因此不需要数据库。


In [4]:
import json
from dataclasses import replace
from nl2sql.knowledge import KnowledgeDoc

raw = next(item for item in json.loads((ROOT / "db/knowledge.json").read_text(encoding="utf-8"))
           if item["id"] == "metric.net_sales")
original = KnowledgeDoc(
    doc_id=raw["id"], kind=raw["kind"], object_ref=raw["object_ref"],
    content=raw["content"], aliases=raw["aliases"], related_tables=raw["related_tables"],
)
revised = replace(original, aliases=[*original.aliases, "退款后收入"])
print({"doc_id": original.doc_id, "原哈希前缀": original.content_hash[:12],
       "新哈希前缀": revised.content_hash[:12], "是否需重新嵌入": original.content_hash != revised.content_hash})
assert original.doc_id == revised.doc_id and original.content_hash != revised.content_hash


{'doc_id': 'metric.net_sales', '原哈希前缀': '490c3f90268c', '新哈希前缀': '94bb56dd8b3b', '是否需重新嵌入': True}


### 三种更新策略的选择

| 策略 | 适用场景 | 取舍 |
|---|---|---|
| 全量重建并覆盖 | 很小的索引、偶发更新 | 实现简单，但每次付出全部嵌入费用；若中途失败需保证旧版仍可用 |
| 哈希增量更新（本项目） | 知识卡 ID 稳定、变更比例低 | 节约在线调用；需正确处理删除、模型更换和同时查询 |
| 新版本构建 → 验收 → 切换 | 不能中断服务、需要回滚 | 更稳妥，但占用额外存储，需版本指针和切流流程 |

注意：本 Notebook 的哈希实验只覆盖指标卡。表卡的实际哈希还包含数据库实时字段，不能只按 <code>knowledge.json</code> 的原始文字计算生产索引哈希。索引器会发在线模型请求并写数据库，**本章没有默认执行它**。


## 7.5 把启动步骤写成分层运行手册

| 顺序 | 命令或检查 | 成功信号 | 失败时优先检查 |
|---|---|---|---|
| 1 | <code>docker compose ps</code> | 数据库容器 healthy | Docker、端口 5432、数据卷 |
| 2 | <code>uv sync --extra dev</code> | 依赖与 Notebook 内核就绪 | Python 版本、虚拟环境 |
| 3 | <code>uv run python -m nl2sql.migrate</code> | <code>analytics</code>、<code>rag</code> 和只读角色存在 | 管理连接、维度 1024；此步写库 |
| 4 | <code>uv run python -m nl2sql.seed</code> | 样例订单 800 行 | 迁移顺序；此步写业务样例 |
| 5 | <code>uv run python -m nl2sql.indexer</code> | 13 张知识卡已索引 | Qwen 密钥、原生 API Host、模型与维度；此步请求在线服务并写索引 |
| 6 | <code>uv run adk web .</code> | 可在开发界面选 <code>nl2sql_agent</code> | DeepSeek 配置、检索或 SQL 工具错误 |

不要用管理连接做运行期健康检查，否则可能掩盖只读角色的授权问题。若 Compose 的密码改了但保留原数据卷，PostgreSQL 已初始化角色的密码不会随配置文件自动改变。上线后应建立独立的数据库迁移、种子/数据导入和索引发布作业，而不是让每个 Agent 进程启动时自动写库。


### 用症状决定下一步，而不是重复执行所有初始化命令

以下是纯 Python 的故障树演算，所有状态均为模拟输入。真实排查应分别记录连接、表、业务数据、索引和模型响应；不要把带凭据的连接字符串写进日志。


In [5]:
def next_check(*, database_up: bool, tables_exist: bool, order_rows: int,
               knowledge_rows: int, embed_model_matches: bool) -> str:
    if not database_up:
        return "检查 Docker 容器、端口与连接"
    if not tables_exist:
        return "检查迁移是否完成与只读角色权限"
    if order_rows == 0:
        return "检查样例数据导入；不要把索引故障误判为无订单"
    if knowledge_rows == 0:
        return "检查索引任务、Qwen 配置与知识源"
    if not embed_model_matches:
        return "核对 QWEN_EMBED_MODEL，并按迁移方案重建索引"
    return "检查在线嵌入、检索命中、SQL 轨迹与最终口径"

examples = [
    dict(database_up=False, tables_exist=False, order_rows=0, knowledge_rows=0, embed_model_matches=False),
    dict(database_up=True, tables_exist=True, order_rows=800, knowledge_rows=0, embed_model_matches=True),
    dict(database_up=True, tables_exist=True, order_rows=800, knowledge_rows=13, embed_model_matches=True),
]
for state in examples:
    print(state, "→", next_check(**state))


{'database_up': False, 'tables_exist': False, 'order_rows': 0, 'knowledge_rows': 0, 'embed_model_matches': False} → 检查 Docker 容器、端口与连接
{'database_up': True, 'tables_exist': True, 'order_rows': 800, 'knowledge_rows': 0, 'embed_model_matches': True} → 检查索引任务、Qwen 配置与知识源
{'database_up': True, 'tables_exist': True, 'order_rows': 800, 'knowledge_rows': 13, 'embed_model_matches': True} → 检查在线嵌入、检索命中、SQL 轨迹与最终口径


## 7.6 可选的只读健康检查与在线检索

第一个开关只连接 <code>DB_READER_URL</code>，读取订单数、知识卡数和当前数据库角色；不修改数据。第二个开关还会通过 <code>nl2sql.retrieval.retrieve()</code> 请求 Qwen 查询向量，并从索引读取文档。运行在线检索前，应先明确自己使用的模型服务额度与网络环境。默认两者均为 <code>False</code>。


In [6]:
if RUN_DB_READ:
    import psycopg
    from nl2sql.config import get_settings

    with psycopg.connect(get_settings().db_reader_url) as conn:
        health = conn.execute("""
            SELECT current_user,
                   (SELECT COUNT(*) FROM analytics.orders),
                   (SELECT COUNT(*) FROM rag.documents)
        """).fetchone()
    print({"数据库角色": health[0], "订单数": health[1], "知识卡数": health[2]})
    assert health[0] == "nl2sql_reader"
else:
    print("已跳过数据库健康检查；需要时将 RUN_DB_READ 设为 True。")

if RUN_ONLINE_RETRIEVAL:
    assert RUN_DB_READ, "在线检索依赖已初始化的数据库；先启用 RUN_DB_READ"
    from nl2sql.retrieval import retrieve

    result = retrieve("2026 年 8 月华东地区净销售额是多少？")
    print("命中文档 ID:", result["doc_ids"])
    assert "metric.net_sales" in result["doc_ids"]
else:
    print("已跳过 Qwen 在线检索；需要时将 RUN_ONLINE_RETRIEVAL 设为 True。")


已跳过数据库健康检查；需要时将 RUN_DB_READ 设为 True。
已跳过 Qwen 在线检索；需要时将 RUN_ONLINE_RETRIEVAL 设为 True。


## 7.7 从教学样例到生产服务，还要补什么？

- **认证与授权**：当前只读数据库角色能读六张业务表，但没有按用户或租户过滤。若接入真实客户数据，检索前要限定可见文档，结构化查询也要有用户级授权；只读不等于人人可读。
- **可观察性**：为每次请求记录脱敏的问题标识、知识文档 ID、索引模型/版本、工具路径、SQL 指纹、各阶段耗时与错误类型；不要记录密钥或不必要的个人原文。监测第 95 百分位延迟（P95, 95th Percentile）、召回率、SQL 成功率和回答正确率。
- **发布与回滚**：模型或知识口径升级时先建新索引，用小型金标准问题集比较必要文档召回、SQL 数值与答案表述，再切换流量；保存可回退的旧版本。
- **恢复能力**：备份数据库，设定恢复演练；把索引构建与 Agent 服务解耦；外部模型暂不可用时明确报错，不能在缺少业务定义时编造金额。
- **打包边界**：当前 <code>knowledge.py</code> 和 <code>migrate.py</code> 通过仓库路径读取 <code>db/</code>，<code>pyproject.toml</code> 的 wheel 配置只列 Python 包。若脱离仓库安装发布包，需把 schema 与知识文件纳入包资源或改为显式配置路径。

这些工作能否上线，取决于实际组织的权限与可靠性要求。技术方案可以不同，但验收必须覆盖“谁能看见什么”和“答案依据是否可追溯”。


## 7.8 综合实践：把一次业务口径变更安全交付

**任务场景**：运营把“净销售额”增加同义词“退款后收入”，并提出“2026 年 8 月华东退款后收入是多少？”请完成一份改造与验证方案。不要直接覆盖旧知识卡后只凭一次成功提问宣布上线。

**基础练习**

1. 在本地知识源找到 <code>metric.net_sales</code>，解释它的稳定 ID、别名、关联表与金额计算口径。
2. 用第 7.4 节的纯函数实验预测变更后的哈希；指出索引器会重新嵌入哪些文档。
3. 写出启动顺序和五项健康检查，注明哪一步读数据库、哪一步写数据库、哪一步调用在线模型。

**检查标准**：修改前后 ID 相同、哈希不同；能准确指出表卡实时字段也可能触发哈希变化；不会在 Notebook、日志或提交记录中泄露凭据；能区分业务表为空和知识索引为空。

**进阶练习**

1. 设计新旧索引并存、离线验收、切流和回滚的发布步骤。至少提出五个问题，包含同义改写、跨地区、跨月、否定条件和无答案请求。
2. 为每题标注必要知识 ID、独立计算的SQL 结果或澄清条件；比较变更前后的前 K 个检索结果召回率（Recall@K, Recall at K）、全部必要知识命中率、端到端正确率、费用与 P95 延迟。
3. 在部署方案表中选择一种落点，补上认证、数据库备份、索引作业调度、访问控制列表（ACL, Access Control List）或等价权限机制、故障告警。写明为什么当前 <code>compose.yaml</code> 还不能直接承担这些职责。

**检查标准**：指标变化有可复现的问题集与独立金标准；能把“召回失败”“SQL 算错”“答案表述错”分开归因；回滚不需要临时重新调用模型重建旧索引。若本地只是完成设计，没有启用写库和在线实验，应明确记录“尚未验证线上行为”。

**课程总结**：业务知识给出定义，检索决定模型看见什么，受限查询提供真实数值，评估与运维决定这条链能否长期可信。
